In [1]:
# Day 5 — Gold Layer Modeling
# Creates dimension tables, fact tables, and KPIs

StatementMeta(, 9ad56fb8-f50c-441a-99fd-28c0ad908c1e, 3, Finished, Available, Finished, False)

##### Creating Dimension Tables

In [8]:
#DIM_CUSTOMERS

from pyspark.sql.functions import col,current_timestamp,lit
from pyspark.sql.types import TimestampType
df_customer = spark.read.table("Ecommerce_lh.silver_customers")

df_dim_customer = df_customer.select(
    "customer_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state"
).withColumn(
    "effective_from",current_timestamp()
).withColumn(
    "effective_to",lit(None).cast(TimestampType())
)

df_dim_customer.write.mode("overwrite").saveAsTable("Ecommerce_lh.dim_customers")

StatementMeta(, 3cc7ad0b-591d-4547-b353-ed79d4b1fbb5, 14, Finished, Available, Finished, False)

##### Applying Merge to handle SCD-2 for Customer data

In [12]:
df_dim_customer.createOrReplaceTempView("temp_customer")

StatementMeta(, bd6128ff-1629-4ffa-b43d-858b796baaac, 14, Finished, Available, Finished, False)

In [ ]:
%%sql
MERGE INTO Ecommerce_lh.dim_customers t
USING temp_customer s
ON t.customer_id = s.customer_id AND t.is_current = 1

WHEN MATCHED AND (
    NOT (t.customer_city <=> s.customer_city) OR
    NOT (t.customer_state <=> s.customer_state)
)
THEN UPDATE SET
    t.is_current = 0,
    t.effective_to = current_timestamp()

In [ ]:
%%sql
INSERT INTO dim_customers (
    customer_id,
    customer_zip_code_prefix,
    customer_city,
    customer_state,
    is_current,
    effective_from,
    effective_to
)
SELECT 
    s.customer_id,
    s.customer_zip_code_prefix,
    s.customer_city,
    s.customer_state,
    1,
    current_timestamp(),
    NULL
FROM temp_customer s
LEFT JOIN dim_customers t
ON s.customer_id = t.customer_id AND t.is_current = 1
WHERE 
    NOT (s.customer_city <=> t.customer_city) 
    OR 
    NOT (s.customer_state <=> t.customer_state)

In [3]:
#DIM_PRODUCTS

df_products = spark.read.table("Ecommerce_lh.silver_products")

df_dim_products = df_products.select(
    "product_id",
    "product_category_name"
)
df_dim_products.write.mode("overwrite").saveAsTable("Ecommerce_lh.dim_products")

StatementMeta(, 9ad56fb8-f50c-441a-99fd-28c0ad908c1e, 5, Finished, Available, Finished, False)

##### Creating Fact Tables

In [4]:
#FACT_ORDERS

df_orders = spark.read.table("Ecommerce_lh.silver_orders_enriched")

df_fact_orders = df_orders.select(
    "order_id",
    "customer_id",
    "order_purchase_timestamp",
    "total_payment_value",
    "order_value",
    "is_delivered"
)

df_fact_orders.write.mode("overwrite").saveAsTable("Ecommerce_lh.fact_orders")

StatementMeta(, 9ad56fb8-f50c-441a-99fd-28c0ad908c1e, 6, Finished, Available, Finished, False)

In [ ]:
#FACT_ORDER_ITEMS

df_silver_orders = spark.read.table("Ecommerce_lh.silver_orders")
df_silver_order_items = spark.read.table("Ecommerce_lh.silver_order_items")

df_join = df_silver_orders.alias("o").join(
    df_silver_order_items.alias("oi"), "order_id","inner"
).select(
    col("oi.order_id"),
    col("oi.product_id"),
    col("o.customer_id"),
    col("o.order_purchase_timestamp"),
    col("oi.price").alias("item_value")
)

df_join.write.mode("overwrite").saveAsTable("Ecommerce_lh.fact_order_items")